In [13]:
# 1. Read all datasets.  
import pandas as pd

customer_df = pd.read_csv("customers.csv")
orders_df = pd.read_csv("orders.csv")
products_df = pd.read_csv("products.csv")
order_items_df = pd.read_csv("order_items.csv")

In [15]:
# 2. Validate schema consistency
customer_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       20000 non-null  int64 
 1   name              20000 non-null  object
 2   email             20000 non-null  object
 3   country           20000 non-null  object
 4   age               20000 non-null  int64 
 5   signup_date       20000 non-null  object
 6   marketing_opt_in  20000 non-null  bool  
dtypes: bool(1), int64(2), object(4)
memory usage: 957.2+ KB


In [4]:
# 3. Detect duplicate customers
customer_no_duplicates_df = customer_df.drop_duplicates()
print(f"Rows after removing duplicates: {len(customer_no_duplicates_df)}")

Rows after removing duplicates: 20000


In [5]:
# 4. Identify invalid records
null_rows_customers = customer_df[customer_df["name"].isna()]
print(null_rows_customers)

Empty DataFrame
Columns: [customer_id, name, email, country, age, signup_date, marketing_opt_in]
Index: []


In [6]:
# 1. Standardize city names (no city name)

In [7]:
# 2. Convert timestamps correctly
orders_df['order_time'] = pd.to_datetime(orders_df['order_time'])

In [ ]:
# 3. Handle missing values
products_df["total_usd"] = products_df["total_usd"].fillna(0)
print(products_df)

In [10]:
# 4. Remove corrupted rows.  
orders_clean_df = orders_df[orders_df['order_time'].notna()]

In [20]:
# 1. Daily sales KPI
# Convert order_time to datetime
orders_df['order_time'] = pd.to_datetime(
    orders_df['order_time'],
    errors='coerce'
)

daily_kpi = (
    orders_df.groupby(orders_df['order_time'].dt.date)
    .agg(
        total_sales=('total_usd', 'sum')
    )
    .reset_index()
)

print(daily_kpi)

      order_time  total_sales
0     2020-01-01      3361.28
1     2020-01-02      2350.86
2     2020-01-03      2163.81
3     2020-01-04      2792.75
4     2020-01-05      2024.13
...          ...          ...
2126  2025-10-27      1582.32
2127  2025-10-28      2462.59
2128  2025-10-29      3174.19
2129  2025-10-30      2928.40
2130  2025-10-31      1765.02

[2131 rows x 2 columns]


In [22]:
# 2. Customer lifetime value
customer_clv = (
    orders_df.groupby('customer_id')
    .agg(
        lifetime_value=('total_usd', 'sum')
    )
    .reset_index()
    .sort_values(by='lifetime_value', ascending=False)
)

print(customer_clv)

       customer_id  lifetime_value
14292        17592         3026.42
301            372         2978.28
14256        17547         2680.64
11473        14135         2588.95
11047        13621         2487.41
...            ...             ...
9485         11660            3.50
7929          9739            3.50
14617        17988            3.32
500            630            3.32
9706         11946            2.80

[16268 rows x 2 columns]


In [ ]:
# 3. Category-wise revenue


In [23]:
# 4.Repeat customer percentage
customer_orders = (
    orders_df.groupby('customer_id')['order_id']
    .count()
    .reset_index(name='total_orders')
)

repeat_customers = customer_orders[
    customer_orders['total_orders'] > 1
]

repeat_customer_percentage = (
    len(repeat_customers) / len(customer_orders)
) * 100

print(f"Repeat Customer Percentage: {repeat_customer_percentage:.2f}%")

Repeat Customer Percentage: 61.75%


In [26]:
#1. Export analytical dataset into csv and parquet
customer_clv.to_csv(
    'customer_lifetime_value.csv',
    index=False
)
customer_clv.to_parquet(
    'customer_lifetime_value.parquet',
    index=False
)
daily_kpi.to_csv(
    'daily_kpi.csv',
    index=False
)
daily_kpi.to_parquet(
    'daily_kpi.parquet',
    index=False
)